# Feature Engineering Matrix

This notebook builds the merged feature matrix for later CPCV and feature selection. It keeps signal-date rows even when an instrument has no market data on that date, leaving missing market features as `NaN`. It does not standardize the dataset, create labels, run CPCV, train models, or calculate feature importance.

## 1. Imports and paths

Paths are resolved so the notebook works whether it is run from the repository root or from inside the `Features` folder.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "raw").exists():
        ROOT = candidate
        break

DATA_DIR = ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
FEATURE_DATA_DIR = DATA_DIR / "features"
HMM_DATA_DIR = FEATURE_DATA_DIR / "hmm"
CODE_FEATURES_DIR = ROOT / "01_feature_engineering" / "technical_analysis"
sys.path.insert(0, str(CODE_FEATURES_DIR))

OHLCV_PATH = RAW_DATA_DIR / "ohlcv_data.csv"
SIGNALS_PATH = RAW_DATA_DIR / "primary_signals.csv"
HMM_CATEGORY_FEATURES_PATH = HMM_DATA_DIR / "probabilities" / "all_category_probabilities.csv"
HMM_LATENT_FEATURES_PATH = HMM_DATA_DIR / "probabilities" / "latent_regime_probabilities.csv"

ALL_TECHNICAL_FEATURES_PATH = FEATURE_DATA_DIR / "all_technical_features.csv"
MERGED_FEATURE_MATRIX_PATH = FEATURE_DATA_DIR / "merged_feature_matrix.csv"
FEATURE_CATALOG_PATH = FEATURE_DATA_DIR / "feature_catalog.csv"
CATEGORICAL_MAPPING_LOG_PATH = FEATURE_DATA_DIR / "categorical_feature_mappings.csv"

## 2. Load source data

OHLCV covers the full historical period, while the primary signals cover the 2020-2022 metamodel window. HMM features are already generated for the signal window.

In [2]:
ohlcv = pd.read_csv(OHLCV_PATH, parse_dates=["date"])
signals = pd.read_csv(SIGNALS_PATH, parse_dates=["date"])
hmm_category_features = pd.read_csv(HMM_CATEGORY_FEATURES_PATH, parse_dates=["date"])
hmm_latent_features = pd.read_csv(HMM_LATENT_FEATURES_PATH, parse_dates=["date"])

for df in [ohlcv, hmm_category_features, hmm_latent_features]:
    df["instrument"] = df["instrument"].str.lower()

ohlcv = ohlcv.sort_values(["instrument", "date"]).reset_index(drop=True)
signals = signals.sort_values("date").reset_index(drop=True)
hmm_category_features = hmm_category_features.sort_values(["instrument", "date"]).reset_index(drop=True)
hmm_latent_features = hmm_latent_features.sort_values(["instrument", "date"]).reset_index(drop=True)

print(f"OHLCV:           {ohlcv.shape} | {ohlcv['date'].min().date()} to {ohlcv['date'].max().date()}")
print(f"Signals:         {signals.shape} | {signals['date'].min().date()} to {signals['date'].max().date()}")
print(f"HMM categories:  {hmm_category_features.shape} | {hmm_category_features['date'].min().date()} to {hmm_category_features['date'].max().date()}")
print(f"HMM latent:      {hmm_latent_features.shape} | {hmm_latent_features['date'].min().date()} to {hmm_latent_features['date'].max().date()}")

OHLCV:           (83547, 8) | 1990-01-02 to 2022-06-30
Signals:         (645, 12) | 2020-01-03 to 2022-06-30
HMM categories:  (7095, 26) | 2020-01-03 to 2022-06-30
HMM latent:      (7095, 9) | 2020-01-03 to 2022-06-30


## 3. Reshape primary signals

The primary signals are supplied wide, with one instrument per column. The merged matrix uses one row per `date + instrument`, so we reshape signals into long format first.

In [3]:
signal_columns = [col for col in signals.columns if col != "date"]

signals_long = signals.melt(
    id_vars="date",
    value_vars=signal_columns,
    var_name="instrument",
    value_name="primary_signal",
)
signals_long["instrument"] = signals_long["instrument"].str.lower()
signals_long = signals_long.sort_values(["instrument", "date"]).reset_index(drop=True)

print(signals_long.shape)
display(signals_long.head())
display(signals_long["primary_signal"].value_counts(dropna=False).sort_index())

(7095, 3)


,date,instrument,primary_signal
0,2020-01-03,cl1s,0
1,2020-01-06,cl1s,0
2,2020-01-07,cl1s,-1
3,2020-01-08,cl1s,0
4,2020-01-09,cl1s,0


primary_signal
-1    1862
 0    2111
 1    3122
Name: count, dtype: int64

## 4. Generate all technical-analysis features

Instead of using the smaller saved `technical_features.csv`, generate the full TA library from `technical_analysis.build_all_features()`. These features are computed on the full OHLCV history before merging to signal dates, so early-2020 rows can use late-2019 lookbacks.

In [4]:
from technical_analysis import build_all_features

technical_features = build_all_features(ohlcv)
technical_features = technical_features.sort_values(["instrument", "date"]).reset_index(drop=True)
technical_features.to_csv(ALL_TECHNICAL_FEATURES_PATH, index=False)

technical_feature_cols = [col for col in technical_features.columns if col not in ["date", "instrument"]]
print(f"Generated technical features: {technical_features.shape}")
print(f"TA feature count: {len(technical_feature_cols)}")
display(technical_features.head())

Generated technical features: (83547, 110)
TA feature count: 108


,date,instrument,log_ret,ret_1d,ret_2d,ret_3d,ret_5d,ret_10d,ret_20d,ret_63d,ret_126d,ret_252d,mom_sign_5d,mom_sign_20d,mom_sign_63d,mom_sign_126d,mom_sign_252d,roc_5d,roc_20d,roc_63d,ret_spread_5_20,ret_spread_20_63,price_vs_sma5,price_vs_sma10,price_vs_sma20,price_vs_sma50,price_vs_sma100,price_vs_sma200,price_vs_ema5,price_vs_ema10,price_vs_ema20,price_vs_ema50,sma5_vs_sma20,sma10_vs_sma50,sma20_vs_sma50,sma50_vs_sma100,sma50_vs_sma200,sma100_vs_sma200,ema5_vs_ema20,ema12_vs_ema26,ema20_vs_ema50,ema20_vs_ema100,sma20_slope,sma50_slope,sma100_slope,ema20_slope,ema50_slope,macd_line,macd_signal,macd_hist,macd_hist_chg,macd_zscore,vol_5d,vol_10d,vol_20d,vol_63d,vol_126d,vol_252d,ewma_vol_5d,ewma_vol_10d,ewma_vol_20d,ewma_vol_63d,vol_ratio_5_20d,vol_ratio_20_63d,vol_ratio_63_126d,park_vol_5d,park_vol_10d,park_vol_20d,park_vol_63d,gk_vol_5d,gk_vol_10d,gk_vol_20d,gk_vol_63d,hl_range,co_range,intraday_ret,overnight_ret,true_range,atr_5d,atr_5d_pct,atr_10d,atr_10d_pct,atr_14d,atr_14d_pct,atr_20d,atr_20d_pct,rsi_7d,rsi_7d_change,rsi_14d,rsi_14d_change,rsi_21d,rsi_21d_change,stoch_k_14d,stoch_d_14d,williams_r_14d,cci_14d,cci_20d,mfi_14d,uo,zscore_20d,zscore_63d,zscore_126d,bb_upper_dist,bb_lower_dist,bb_position,bb_width,bb_width_zscore,donchian_pos_20d,donchian_pos_52d,donchian_pos_252d
0,1990-01-02,cl1s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.13,1.09,0.048790,NaN,1.13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1990-01-03,cl1s,0.033931,0.034513,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.022747,0.028062,0.031124,0.033115,NaN,NaN,NaN,NaN,NaN,NaN,0.008190,0.002746,0.001931,0.002602,NaN,NaN,NaN,0.003287,0.001353,0.063020,0.012604,0.050416,0.050416,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.80,0.48,0.020479,0.013452,0.91,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1990-01-04,cl1s,-0.011468,-0.011402,0.022717,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.007363,0.013329,0.017490,0.020481,NaN,NaN,NaN,NaN,NaN,NaN,0.010053,0.003922,0.002940,0.004013,NaN,NaN,NaN,0.001844,0.000837,0.090138,0.028111,0.062027,0.011611,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.09,-0.47,-0.019878,0.008410,1.09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1990-01-05,cl1s,-0.014197,-0.014097,-0.025338,0.008301,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.004569,-0.000782,0.002846,0.005856,NaN,NaN,NaN,NaN,NaN,NaN,0.007449,0.003655,0.003001,0.004172,NaN,NaN,NaN,0.000300,0.000239,0.084032,0.039295,0.044737,-0.017290,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.67,-0.34,-0.014624,0.000427,0.67,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1990-01-08,cl1s,-0.065348,-0.063258,-0.076463,-0.086993,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.057377,NaN,NaN,NaN,NaN,NaN,-0.046062,-0.052972,-0.055139,-0.055634,NaN,NaN,NaN,NaN,NaN,NaN,-0.009516,-0.001668,-0.000523,-0.000502,NaN,NaN,NaN,-0.005771,-0.002266,-0.038176,0.023801,-0.061977,-0.106714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.404166,NaN,NaN,NaN,0.34903,NaN,NaN,NaN,1.05,-0.98,-0.044331,-0.021017,1.53,1.108752,0.051284,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

## 5. Build full-history market features

These features use volume, open interest, and candle structure. They are also computed from the full OHLCV history before filtering to the signal window.

In [5]:
def rolling_zscore(series: pd.Series, window: int) -> pd.Series:
    rolling_mean = series.rolling(window=window, min_periods=window).mean()
    rolling_std = series.rolling(window=window, min_periods=window).std()
    return (series - rolling_mean) / rolling_std.replace(0, np.nan)


def build_market_features(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values("date").copy()
    safe_volume = group["volume"].where(group["volume"] > 0)
    safe_open_interest = group["open_interest"].where(group["open_interest"] > 0)
    safe_open = group["open"].replace(0, np.nan)
    safe_close = group["close"].replace(0, np.nan)
    safe_range = (group["high"] - group["low"]).replace(0, np.nan)
    prev_close = group["close"].shift(1)

    out = group[["date", "instrument"]].copy()
    out["volume_log_change"] = np.log(safe_volume).diff()
    out["open_interest_log_change"] = np.log(safe_open_interest).diff()
    out["volume_zscore_20d"] = rolling_zscore(group["volume"], 20)
    out["volume_zscore_63d"] = rolling_zscore(group["volume"], 63)
    out["open_interest_zscore_20d"] = rolling_zscore(group["open_interest"], 20)
    out["open_interest_zscore_63d"] = rolling_zscore(group["open_interest"], 63)
    out["volume_to_open_interest"] = group["volume"] / group["open_interest"].replace(0, np.nan)

    out["range_pct"] = (group["high"] - group["low"]) / safe_close
    out["body_pct"] = (group["close"] - group["open"]) / safe_open
    out["upper_wick_pct"] = (group["high"] - group[["open", "close"]].max(axis=1)) / safe_close
    out["lower_wick_pct"] = (group[["open", "close"]].min(axis=1) - group["low"]) / safe_close
    out["close_position_in_bar"] = (group["close"] - group["low"]) / safe_range
    out["gap_open_pct"] = group["open"] / prev_close.replace(0, np.nan) - 1

    out["_ret_for_vol"] = group["close"].pct_change()
    out["vol_20d_for_interaction"] = out["_ret_for_vol"].rolling(20, min_periods=20).std()
    return out.drop(columns="_ret_for_vol")


market_features = (
    ohlcv.groupby("instrument", group_keys=False)
    .apply(build_market_features)
    .replace([np.inf, -np.inf], np.nan)
    .reset_index(drop=True)
)

market_feature_cols = [col for col in market_features.columns if col not in ["date", "instrument"]]
print(market_features.shape)
display(market_features.head())

(83547, 16)


C:\Users\madhu\AppData\Local\Temp\claude\ipykernel_49972\2505992879.py:39: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_market_features)


,date,instrument,volume_log_change,open_interest_log_change,volume_zscore_20d,volume_zscore_63d,open_interest_zscore_20d,open_interest_zscore_63d,volume_to_open_interest,range_pct,body_pct,upper_wick_pct,lower_wick_pct,close_position_in_bar,gap_open_pct,vol_20d_for_interaction
0,1990-01-02,cl1s,NaN,NaN,NaN,NaN,NaN,NaN,0.344875,0.049367,0.050000,0.001311,0.000437,0.973451,NaN,NaN
1,1990-01-03,cl1s,0.680850,-0.076445,NaN,NaN,NaN,NaN,0.735446,0.033784,0.020690,0.005068,0.008446,0.850000,0.013543,NaN
2,1990-01-04,cl1s,0.102654,-0.007074,NaN,NaN,NaN,NaN,0.820739,0.046561,-0.019682,0.001709,0.024776,0.532110,0.008446,NaN
3,1990-01-05,cl1s,0.058370,-0.063225,NaN,NaN,NaN,NaN,0.926857,0.029029,-0.014518,0.012132,0.002166,0.074627,0.000427,NaN
4,1990-01-08,cl1s,-0.289757,-0.046728,NaN,NaN,NaN,NaN,0.726887,0.048566,-0.043363,0.000000,0.003238,0.066667,-0.020797,NaN


## 6. Build signal-behaviour features

These describe the primary model's recent behaviour on the signal calendar: whether the signal just changed, how long the current signal has persisted, and recent signal stability.

In [6]:
def build_signal_behaviour_features(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values("date").copy()
    signal_changed = group["primary_signal"].ne(group["primary_signal"].shift(1))
    if len(signal_changed) > 0:
        signal_changed.iloc[0] = False
    signal_run_id = signal_changed.cumsum()

    out = group[["date", "instrument"]].copy()
    out["signal_changed"] = signal_changed.astype(int)
    out["days_since_signal_change"] = group.groupby(signal_run_id).cumcount()
    out["signal_rolling_mean_5d"] = group["primary_signal"].rolling(5, min_periods=1).mean()
    out["signal_rolling_mean_20d"] = group["primary_signal"].rolling(20, min_periods=1).mean()
    out["signal_abs_rolling_sum_20d"] = group["primary_signal"].abs().rolling(20, min_periods=1).sum()
    return out


signal_features = (
    signals_long.groupby("instrument", group_keys=False)
    .apply(build_signal_behaviour_features)
    .reset_index(drop=True)
)

signal_feature_cols = [col for col in signal_features.columns if col not in ["date", "instrument"]]
print(signal_features.shape)
display(signal_features.head())

(7095, 7)


C:\Users\madhu\AppData\Local\Temp\claude\ipykernel_49972\424644578.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_signal_behaviour_features)


,date,instrument,signal_changed,days_since_signal_change,signal_rolling_mean_5d,signal_rolling_mean_20d,signal_abs_rolling_sum_20d
0,2020-01-03,cl1s,0,0,0.000000,0.000000,0.0
1,2020-01-06,cl1s,0,1,0.000000,0.000000,0.0
2,2020-01-07,cl1s,1,0,-0.333333,-0.333333,1.0
3,2020-01-08,cl1s,1,0,-0.250000,-0.250000,1.0
4,2020-01-09,cl1s,0,1,-0.200000,-0.200000,1.0


## 7. Merge all feature sets

Start from `signals_long` so every primary-signal row is retained, including instrument-level non-trading rows. Market and technical features stay `NaN` when that instrument has no OHLCV row on the signal date.

In [7]:
hmm_category_merge = hmm_category_features.drop(columns=["primary_signal"], errors="ignore")
latent_probability_cols = [col for col in hmm_latent_features.columns if col.startswith("hmm_regime_prob_")]
hmm_latent_probabilities = hmm_latent_features[["date", "instrument", *latent_probability_cols]].copy()

assert not signals_long.duplicated(["date", "instrument"]).any(), "Signals contain duplicate date/instrument keys."
assert not hmm_category_merge.duplicated(["date", "instrument"]).any(), "HMM category features contain duplicate date/instrument keys."
assert not hmm_latent_probabilities.duplicated(["date", "instrument"]).any(), "HMM latent probabilities contain duplicate date/instrument keys."

feature_matrix = (
    signals_long
    .merge(ohlcv, on=["date", "instrument"], how="left", validate="one_to_one")
    .merge(technical_features, on=["date", "instrument"], how="left", validate="one_to_one")
    .merge(hmm_category_merge, on=["date", "instrument"], how="left", validate="one_to_one")
    .merge(hmm_latent_probabilities, on=["date", "instrument"], how="left", validate="one_to_one")
    .merge(market_features, on=["date", "instrument"], how="left", validate="one_to_one")
    .merge(signal_features, on=["date", "instrument"], how="left", validate="one_to_one")
)

feature_matrix = feature_matrix.sort_values(["instrument", "date"]).reset_index(drop=True)
feature_matrix = feature_matrix.replace([np.inf, -np.inf], np.nan)

print(feature_matrix.shape)
display(feature_matrix.head())

(7095, 163)


,date,instrument,primary_signal,open,high,low,close,volume,open_interest,log_ret,ret_1d,ret_2d,ret_3d,ret_5d,ret_10d,ret_20d,ret_63d,ret_126d,ret_252d,mom_sign_5d,mom_sign_20d,mom_sign_63d,mom_sign_126d,mom_sign_252d,roc_5d,roc_20d,roc_63d,ret_spread_5_20,ret_spread_20_63,price_vs_sma5,price_vs_sma10,price_vs_sma20,price_vs_sma50,price_vs_sma100,price_vs_sma200,price_vs_ema5,price_vs_ema10,price_vs_ema20,price_vs_ema50,sma5_vs_sma20,sma10_vs_sma50,sma20_vs_sma50,sma50_vs_sma100,sma50_vs_sma200,sma100_vs_sma200,ema5_vs_ema20,ema12_vs_ema26,ema20_vs_ema50,ema20_vs_ema100,sma20_slope,sma50_slope,sma100_slope,ema20_slope,ema50_slope,macd_line,macd_signal,macd_hist,macd_hist_chg,macd_zscore,vol_5d,vol_10d,vol_20d,vol_63d,vol_126d,vol_252d,ewma_vol_5d,ewma_vol_10d,ewma_vol_20d,ewma_vol_63d,vol_ratio_5_20d,...,rsi_7d,rsi_7d_change,rsi_14d,rsi_14d_change,rsi_21d,rsi_21d_change,stoch_k_14d,stoch_d_14d,williams_r_14d,cci_14d,cci_20d,mfi_14d,uo,zscore_20d,zscore_63d,zscore_126d,bb_upper_dist,bb_lower_dist,bb_position,bb_width,bb_width_zscore,donchian_pos_20d,donchian_pos_52d,donchian_pos_252d,hmm_predicted_regime,hmm_volatility_regime,hmm_return_regime,hmm_market_state,hmm_regime_confidence,hmm_prob_low_vol,hmm_prob_normal_vol,hmm_prob_high_vol,hmm_prob_extreme_vol,hmm_prob_downside,hmm_prob_weak,hmm_prob_positive,hmm_prob_strong_upside,hmm_prob_stress,hmm_prob_upside_breakout,hmm_prob_calm_positive,hmm_prob_calm_negative,hmm_prob_chop,hmm_prob_high_or_extreme_vol,hmm_prob_low_or_normal_vol,hmm_prob_downside_or_weak,hmm_prob_positive_or_strong_upside,hmm_prob_not_stress,hmm_regime_prob_0,hmm_regime_prob_1,hmm_regime_prob_2,hmm_regime_prob_3,volume_log_change,open_interest_log_change,volume_zscore_20d,volume_zscore_63d,open_interest_zscore_20d,open_interest_zscore_63d,volume_to_open_interest,range_pct,body_pct,upper_wick_pct,lower_wick_pct,close_position_in_bar,gap_open_pct,vol_20d_for_interaction,signal_changed,days_since_signal_change,signal_rolling_mean_5d,signal_rolling_mean_20d,signal_abs_rolling_sum_20d
0,2020-01-03,cl1s,0,24.795579,25.974970,24.775315,25.553469,2.185752e+06,958523.501762,0.030108,0.030566,0.032591,0.022211,0.022211,0.036154,0.080688,0.199550,0.095344,0.299427,1.0,1.0,1.0,1.0,1.0,0.022211,0.080688,0.199550,-0.058477,-0.118861,0.021251,0.027509,0.043218,0.083466,0.110436,0.086984,0.019371,0.027635,0.042866,0.073641,0.021510,0.054459,0.038581,0.024892,0.003247,-0.021120,0.023048,0.019468,0.029510,0.046538,0.003910,0.002946,0.001444,0.004533,0.003015,0.473381,0.443450,0.029931,0.032402,1.310175,0.240319,0.186821,0.152724,0.246618,0.371613,0.337063,0.272244,0.223764,0.219735,0.285221,1.573551,...,78.102827,18.324875,70.179907,8.628436,65.494265,5.618253,78.813566,72.541698,-21.186434,205.675985,158.387285,69.962898,60.871319,2.108372,2.135498,2.536731,-0.002129,0.080726,1.027093,0.081993,-0.967451,0.839418,0.907283,0.781953,3.0,high_vol,strong_upside,upside_breakout,0.991140,7.946343e-04,7.739873e-03,9.911404e-01,0.000325,0.000325,7.946343e-04,7.739873e-03,9.911404e-01,0.000325,9.911404e-01,7.739873e-03,7.946343e-04,0.0,0.991465,8.534508e-03,0.001120,9.988803e-01,9.996749e-01,0.000325,7.739873e-03,7.946343e-04,9.911404e-01,0.598557,-0.022820,2.469962,2.781207,0.174642,0.162133,2.280332,0.046947,0.030566,0.016495,0.000793,0.648649,0.000000,0.009692,0,0,0.000000,0.000000,0.0
1,2020-01-06,cl1s,0,25.820960,26.230302,25.387301,25.642633,1.786962e+06,909240.146872,0.003483,0.003489,0.034161,0.036194,0.025113,0.034161,0.084459,0.195529,0.096306,0.280307,1.0,1.0,1.0,1.0,1.0,0.025113,0.084459,0.195529,-0.059346,-0.111070,0.019694,0.027594,0.042608,0.084535,0.113088,0.090535,0.015170,0.025400,0.041891,0.074127,0.022471,0.055412,0.040214,0.026328,0.005532,-0.020262,0.026322,0.020739,0.030941,0.049195,0.004077,0.002501,0.001099,0.004429,0.003035,0.506348,0.456029,0.050319,0.020387,1.392643,0.238370,0.186535,0.152049,0.246491,0.371625,0.336626,0.228189,0.203104,0.209062,0.280741,1.567717,...,79.391523,1.288696,71.004332,0.824425,66.

## 8. Add HMM numeric encodings and interactions

Missing HMM regimes stay visibly missing: one-hot columns are `NaN` when the underlying latent/category label is missing. The interpreted HMM category labels are converted into numeric `_code` columns, and their string-to-code mappings are saved to `data/features/categorical_feature_mappings.csv` so the transformation is auditable. The original string label columns are removed from the final matrix after encoding.

In [8]:
hmm_encoding_feature_cols = []
hmm_category_code_cols = []
categorical_mapping_rows = []

feature_matrix["hmm_regime_missing"] = feature_matrix["hmm_predicted_regime"].isna().astype(int)
hmm_encoding_feature_cols.append("hmm_regime_missing")

for state in range(4):
    col = f"hmm_regime_{state}"
    feature_matrix[col] = np.nan
    known = feature_matrix["hmm_predicted_regime"].notna()
    feature_matrix.loc[known, col] = feature_matrix.loc[known, "hmm_predicted_regime"].eq(state).astype(int)
    hmm_encoding_feature_cols.append(col)


def add_category_code(source_col: str, code_col: str, mapping: dict[str, int], mapping_type: str) -> str:
    feature_matrix[code_col] = feature_matrix[source_col].map(mapping).astype("float64")
    hmm_category_code_cols.append(code_col)
    for label, code in mapping.items():
        categorical_mapping_rows.append(
            {
                "source_column": source_col,
                "encoded_column": code_col,
                "category_label": label,
                "numeric_value": code,
                "mapping_type": mapping_type,
                "missing_value": "NaN",
                "notes": "Original string labels are removed from merged_feature_matrix.csv after this encoding.",
            }
        )
    return code_col


hmm_category_code_mappings = {
    "hmm_volatility_regime": {
        "code_col": "hmm_volatility_regime_code",
        "mapping_type": "ordered_low_to_high_volatility",
        "mapping": {"low_vol": 0, "normal_vol": 1, "high_vol": 2, "extreme_vol": 3},
    },
    "hmm_return_regime": {
        "code_col": "hmm_return_regime_code",
        "mapping_type": "ordered_negative_to_positive_return",
        "mapping": {"downside": 0, "weak": 1, "positive": 2, "strong_upside": 3},
    },
    "hmm_market_state": {
        "code_col": "hmm_market_state_code",
        "mapping_type": "nominal_deterministic_code",
        "mapping": {"stress": 0, "upside_breakout": 1, "calm_positive": 2, "calm_negative": 3, "chop": 4},
    },
}

for source_col, spec in hmm_category_code_mappings.items():
    add_category_code(source_col, spec["code_col"], spec["mapping"], spec["mapping_type"])


def add_nullable_one_hot(source_col: str, prefix: str, labels: list[str]) -> list[str]:
    missing_col = f"{source_col}_missing"
    feature_matrix[missing_col] = feature_matrix[source_col].isna().astype(int)
    created = [missing_col]
    known = feature_matrix[source_col].notna()
    for label in labels:
        col = f"{prefix}_{label}"
        feature_matrix[col] = np.nan
        feature_matrix.loc[known, col] = feature_matrix.loc[known, source_col].eq(label).astype(int)
        created.append(col)
    return created


hmm_encoding_feature_cols.extend(
    add_nullable_one_hot(
        "hmm_volatility_regime",
        "hmm_volatility",
        ["low_vol", "normal_vol", "high_vol", "extreme_vol"],
    )
)
hmm_encoding_feature_cols.extend(
    add_nullable_one_hot(
        "hmm_return_regime",
        "hmm_return",
        ["downside", "weak", "positive", "strong_upside"],
    )
)
hmm_encoding_feature_cols.extend(
    add_nullable_one_hot(
        "hmm_market_state",
        "hmm_market",
        ["stress", "upside_breakout", "calm_positive", "calm_negative", "chop"],
    )
)

feature_matrix["signal_x_hmm_confidence"] = feature_matrix["primary_signal"] * feature_matrix["hmm_regime_confidence"]
feature_matrix["ret_5d_x_hmm_confidence"] = feature_matrix["ret_5d"] * feature_matrix["hmm_regime_confidence"]
feature_matrix["vol_20d_x_hmm_confidence"] = feature_matrix["vol_20d_for_interaction"] * feature_matrix["hmm_regime_confidence"]
feature_matrix["signal_x_hmm_prob_stress"] = feature_matrix["primary_signal"] * feature_matrix["hmm_prob_stress"]
feature_matrix["signal_x_hmm_prob_high_or_extreme_vol"] = feature_matrix["primary_signal"] * feature_matrix["hmm_prob_high_or_extreme_vol"]
feature_matrix["signal_x_hmm_prob_positive_or_strong_upside"] = feature_matrix["primary_signal"] * feature_matrix["hmm_prob_positive_or_strong_upside"]

interaction_feature_cols = [
    "signal_x_hmm_confidence",
    "ret_5d_x_hmm_confidence",
    "vol_20d_x_hmm_confidence",
    "signal_x_hmm_prob_stress",
    "signal_x_hmm_prob_high_or_extreme_vol",
    "signal_x_hmm_prob_positive_or_strong_upside",
]

categorical_mapping_log = pd.DataFrame(categorical_mapping_rows)
feature_matrix = feature_matrix.drop(columns=list(hmm_category_code_mappings.keys()))

print(feature_matrix[hmm_category_code_cols + hmm_encoding_feature_cols + interaction_feature_cols].head())
display(categorical_mapping_log)

   hmm_volatility_regime_code  hmm_return_regime_code  hmm_market_state_code  hmm_regime_missing  hmm_regime_0  hmm_regime_1  hmm_regime_2  hmm_regime_3  \
0                         2.0                     3.0                    1.0                   0           0.0           0.0           0.0           1.0   
1                         0.0                     1.0                    3.0                   0           0.0           0.0           1.0           0.0   
2                         0.0                     1.0                    3.0                   0           0.0           0.0           1.0           0.0   
3                         3.0                     0.0                    0.0                   0           1.0           0.0           0.0           0.0   
4                         0.0                     1.0                    3.0                   0           0.0           0.0           1.0           0.0   

   hmm_volatility_regime_missing  hmm_volatility_low_vol  hmm_v

,source_column,encoded_column,category_label,numeric_value,mapping_type,missing_value,notes
0,hmm_volatility_regime,hmm_volatility_regime_code,low_vol,0,ordered_low_to_high_volatility,NaN,Original string labels are removed from merged...
1,hmm_volatility_regime,hmm_volatility_regime_code,normal_vol,1,ordered_low_to_high_volatility,NaN,Original string labels are removed from merged...
2,hmm_volatility_regime,hmm_volatility_regime_code,high_vol,2,ordered_low_to_high_volatility,NaN,Original string labels are removed from merged...
3,hmm_volatility_regime,hmm_volatility_regime_code,extreme_vol,3,ordered_low_to_high_volatility,NaN,Original string labels are removed from merged...
4,hmm_return_regime,hmm_return_regime_code,downside,0,ordered_negative_to_positive_return,NaN,Original string labels are removed from merged...
5,hmm_return_regime,hmm_return_regime_code,weak,1,ordered_negative_to_positive_return,NaN,Original string labels are removed from merged...
6,hmm_return_regime,hmm_return_regime_code,positive,2,ordered_negative_to_positive_return,NaN,Original string labels are removed from merged...
7,hmm_return_regime,hmm_return_regime_code,strong_upside,3,ordered_negative_to_positive_return,NaN,Original string labels are removed from merged...
8,hmm_market_state,hmm_market_state_code,stress,0,nominal_deterministic_code,NaN,Original string labels are removed from merged...
9,hmm_market_state,hmm_market_state_code,upside_breakout,1,nominal_deterministic_code,NaN,Original string labels are removed from merged...


## 9. Validate the matrix

The checks confirm the full signal calendar is retained, generated TA features are present, early-2020 features have full-history lookbacks where market data exists, and holiday-specific features are absent for now.

In [9]:
duplicate_keys = feature_matrix.duplicated(["date", "instrument"]).sum()
numeric = feature_matrix.select_dtypes(include=[np.number])
infinite_values = np.isinf(numeric).sum().sum()
missing_ta_cols = [col for col in technical_feature_cols if col not in feature_matrix.columns]
expected_category_probability_cols = [
    "hmm_prob_low_vol",
    "hmm_prob_normal_vol",
    "hmm_prob_high_vol",
    "hmm_prob_extreme_vol",
    "hmm_prob_downside",
    "hmm_prob_weak",
    "hmm_prob_positive",
    "hmm_prob_strong_upside",
    "hmm_prob_stress",
    "hmm_prob_upside_breakout",
    "hmm_prob_calm_positive",
    "hmm_prob_calm_negative",
    "hmm_prob_chop",
    "hmm_prob_high_or_extreme_vol",
    "hmm_prob_low_or_normal_vol",
    "hmm_prob_downside_or_weak",
    "hmm_prob_positive_or_strong_upside",
    "hmm_prob_not_stress",
]
expected_category_label_cols = []
expected_category_code_cols = ["hmm_volatility_regime_code", "hmm_return_regime_code", "hmm_market_state_code"]
expected_latent_probability_cols = [f"hmm_regime_prob_{state}" for state in range(4)]
expected_hmm_encoding_cols = [
    "hmm_regime_missing",
    "hmm_regime_0",
    "hmm_regime_1",
    "hmm_regime_2",
    "hmm_regime_3",
    "hmm_volatility_regime_missing",
    "hmm_return_regime_missing",
    "hmm_market_state_missing",
    "hmm_volatility_low_vol",
    "hmm_volatility_normal_vol",
    "hmm_volatility_high_vol",
    "hmm_volatility_extreme_vol",
    "hmm_return_downside",
    "hmm_return_weak",
    "hmm_return_positive",
    "hmm_return_strong_upside",
    "hmm_market_stress",
    "hmm_market_upside_breakout",
    "hmm_market_calm_positive",
    "hmm_market_calm_negative",
    "hmm_market_chop",
]
missing_hmm_cols = [
    col
    for col in [
        "hmm_predicted_regime",
        "hmm_regime_confidence",
        *expected_category_label_cols,
        *expected_category_code_cols,
        *expected_category_probability_cols,
        *expected_latent_probability_cols,
        *expected_hmm_encoding_cols,
    ]
    if col not in feature_matrix.columns
]
holiday_specific_cols = [
    "has_market_data",
    "is_instrument_holiday",
    "is_partial_market_holiday",
    "missing_market_data_count",
    "days_to_next_holiday",
    "days_since_last_holiday",
    "signal_rows_to_next_holiday",
    "signal_rows_since_last_holiday",
]
present_holiday_cols = [col for col in holiday_specific_cols if col in feature_matrix.columns]

validation = pd.Series(
    {
        "rows": len(feature_matrix),
        "columns": feature_matrix.shape[1],
        "date_min": feature_matrix["date"].min(),
        "date_max": feature_matrix["date"].max(),
        "instrument_count": feature_matrix["instrument"].nunique(),
        "technical_feature_count": len(technical_feature_cols),
        "missing_ta_columns": len(missing_ta_cols),
        "missing_hmm_columns": len(missing_hmm_cols),
        "hmm_category_source_rows": len(hmm_category_features),
        "hmm_latent_source_rows": len(hmm_latent_features),
        "hmm_category_source_duplicate_keys": hmm_category_features.duplicated(["date", "instrument"]).sum(),
        "hmm_latent_source_duplicate_keys": hmm_latent_features.duplicated(["date", "instrument"]).sum(),
        "holiday_specific_columns_present": len(present_holiday_cols),
        "duplicate_date_instrument_keys": duplicate_keys,
        "infinite_numeric_values": infinite_values,
        "missing_close_rows": feature_matrix["close"].isna().sum(),
    }
)

display(validation)
display(feature_matrix.loc[feature_matrix["date"].eq("2020-02-17"), ["date", "instrument", "close", "volume_zscore_63d", "hmm_regime_missing", "hmm_market_state_code", "hmm_prob_stress"]].sort_values("instrument"))
display(feature_matrix.isna().mean().sort_values(ascending=False).head(30).to_frame("missing_pct"))

assert len(technical_feature_cols) == 108, "Expected 108 generated TA feature columns."
assert len(missing_ta_cols) == 0, "Some generated TA columns are missing from the matrix."
assert len(missing_hmm_cols) == 0, f"Some expected HMM columns are missing from the matrix: {missing_hmm_cols}"
assert len(categorical_mapping_log) == 13, "Unexpected number of categorical mapping rows."
assert not [col for col in feature_matrix.select_dtypes(exclude=[np.number]).columns if col not in ["date", "instrument"]], "Non-numeric feature columns remain in the matrix."
assert len(hmm_category_features) == len(signals_long), "HMM category source should match signal-date row count."
assert len(hmm_latent_features) == len(signals_long), "HMM latent source should match signal-date row count."
assert not hmm_category_features.duplicated(["date", "instrument"]).any(), "HMM category source has duplicate keys."
assert not hmm_latent_features.duplicated(["date", "instrument"]).any(), "HMM latent source has duplicate keys."
assert len(feature_matrix) == len(signals_long), "The matrix should retain all signal-date rows."
assert feature_matrix["date"].min() == signals_long["date"].min(), "Unexpected matrix start date."
assert feature_matrix["date"].max() == signals_long["date"].max(), "Unexpected matrix end date."
assert feature_matrix["instrument"].nunique() == len(signal_columns), "Unexpected number of instruments."
assert duplicate_keys == 0, "Duplicate date/instrument keys found."
assert infinite_values == 0, "Infinite values found in numeric columns."
assert len(present_holiday_cols) == 0, f"Holiday-specific columns should be absent: {present_holiday_cols}"
assert len(feature_matrix[feature_matrix["date"].eq("2020-02-17")]) == 11, "Expected 11 rows on 2020-02-17."
assert feature_matrix.loc[(feature_matrix["date"].eq("2020-02-17")) & (feature_matrix["instrument"].eq("fesx1s")), "close"].notna().all(), "FESX1S should have market data on 2020-02-17."
assert feature_matrix.loc[(feature_matrix["date"].eq("2020-02-17")) & (~feature_matrix["instrument"].eq("fesx1s")), "close"].isna().all(), "Other instruments should retain missing market data on 2020-02-17."
assert feature_matrix.loc[feature_matrix["date"].eq("2020-01-03"), "ret_252d"].notna().all(), "Early-2020 TA lookbacks should use pre-2020 history."
assert feature_matrix.loc[feature_matrix["date"].eq("2020-01-03"), "vol_20d_for_interaction"].notna().all(), "Early-2020 interaction volatility should use pre-2020 history."

display(feature_matrix.sample(5, random_state=42))

rows                                                 7095
columns                                               190
date_min                              2020-01-03 00:00:00
date_max                              2022-06-30 00:00:00
instrument_count                                       11
technical_feature_count                               108
missing_ta_columns                                      0
missing_hmm_columns                                     0
hmm_category_source_rows                             7095
hmm_latent_source_rows                               7095
hmm_category_source_duplicate_keys                      0
hmm_latent_source_duplicate_keys                        0
holiday_specific_columns_present                        0
duplicate_date_instrument_keys                          0
infinite_numeric_values                                 0
missing_close_rows                                    176
dtype: object

,date,instrument,close,volume_zscore_63d,hmm_regime_missing,hmm_market_state_code,hmm_prob_stress
31,2020-02-17,cl1s,NaN,NaN,1,NaN,NaN
676,2020-02-17,es1s,NaN,NaN,1,NaN,NaN
1321,2020-02-17,fesx1s,5382.310031,-1.379652,0,2.0,3.032957e-07
1966,2020-02-17,gc1s,NaN,NaN,1,NaN,NaN
2611,2020-02-17,hg1s,NaN,NaN,1,NaN,NaN
3256,2020-02-17,ho1s,NaN,NaN,1,NaN,NaN
3901,2020-02-17,ng1s,NaN,NaN,1,NaN,NaN
4546,2020-02-17,nq1s,NaN,NaN,1,NaN,NaN
5191,2020-02-17,pl1s,NaN,NaN,1,NaN,NaN
5836,2020-02-17,rb1s,NaN,NaN,1,NaN,NaN


,missing_pct
signal_x_hmm_confidence,0.02537
ret_5d_x_hmm_confidence,0.02537
signal_x_hmm_prob_high_or_extreme_vol,0.02537
hmm_market_upside_breakout,0.02537
hmm_market_calm_positive,0.02537
hmm_market_calm_negative,0.02537
signal_x_hmm_prob_positive_or_strong_upside,0.02537
signal_x_hmm_prob_stress,0.02537
hmm_return_downside,0.02537
hmm_regime_prob_0,0.02537


,date,instrument,primary_signal,open,high,low,close,volume,open_interest,log_ret,ret_1d,ret_2d,ret_3d,ret_5d,ret_10d,ret_20d,ret_63d,ret_126d,ret_252d,mom_sign_5d,mom_sign_20d,mom_sign_63d,mom_sign_126d,mom_sign_252d,roc_5d,roc_20d,roc_63d,ret_spread_5_20,ret_spread_20_63,price_vs_sma5,price_vs_sma10,price_vs_sma20,price_vs_sma50,price_vs_sma100,price_vs_sma200,price_vs_ema5,price_vs_ema10,price_vs_ema20,price_vs_ema50,sma5_vs_sma20,sma10_vs_sma50,sma20_vs_sma50,sma50_vs_sma100,sma50_vs_sma200,sma100_vs_sma200,ema5_vs_ema20,ema12_vs_ema26,ema20_vs_ema50,ema20_vs_ema100,sma20_slope,sma50_slope,sma100_slope,ema20_slope,ema50_slope,macd_line,macd_signal,macd_hist,macd_hist_chg,macd_zscore,vol_5d,vol_10d,vol_20d,vol_63d,vol_126d,vol_252d,ewma_vol_5d,ewma_vol_10d,ewma_vol_20d,ewma_vol_63d,vol_ratio_5_20d,...,hmm_prob_normal_vol,hmm_prob_high_vol,hmm_prob_extreme_vol,hmm_prob_downside,hmm_prob_weak,hmm_prob_positive,hmm_prob_strong_upside,hmm_prob_stress,hmm_prob_upside_breakout,hmm_prob_calm_positive,hmm_prob_calm_negative,hmm_prob_chop,hmm_prob_high_or_extreme_vol,hmm_prob_low_or_normal_vol,hmm_prob_downside_or_weak,hmm_prob_positive_or_strong_upside,hmm_prob_not_stress,hmm_regime_prob_0,hmm_regime_prob_1,hmm_regime_prob_2,hmm_regime_prob_3,volume_log_change,open_interest_log_change,volume_zscore_20d,volume_zscore_63d,open_interest_zscore_20d,open_interest_zscore_63d,volume_to_open_interest,range_pct,body_pct,upper_wick_pct,lower_wick_pct,close_position_in_bar,gap_open_pct,vol_20d_for_interaction,signal_changed,days_since_signal_change,signal_rolling_mean_5d,signal_rolling_mean_20d,signal_abs_rolling_sum_20d,hmm_regime_missing,hmm_regime_0,hmm_regime_1,hmm_regime_2,hmm_regime_3,hmm_volatility_regime_code,hmm_return_regime_code,hmm_market_state_code,hmm_volatility_regime_missing,hmm_volatility_low_vol,hmm_volatility_normal_vol,hmm_volatility_high_vol,hmm_volatility_extreme_vol,hmm_return_regime_missing,hmm_return_downside,hmm_return_weak,hmm_return_positive,hmm_return_strong_upside,hmm_market_state_missing,hmm_market_stress,hmm_market_upside_breakout,hmm_market_calm_positive,hmm_market_calm_negative,hmm_market_chop,signal_x_hmm_confidence,ret_5d_x_hmm_confidence,vol_20d_x_hmm_confidence,signal_x_hmm_prob_stress,signal_x_hmm_prob_high_or_extreme_vol,signal_x_hmm_prob_positive_or_strong_upside
1024,2021-06-22,es1s,1,3676.737142,3699.611907,3664.972978,3691.551276,1.220708e+06,2.833070e+06,0.005325,0.005340,0.019923,0.005698,-0.000059,0.004610,0.012275,0.093920,0.154487,0.371078,-1.0,1.0,1.0,1.0,1.0,-0.000059,0.012275,0.093920,-0.012334,-0.081645,0.007252,0.004175,0.006798,0.015316,0.048658,0.118390,0.005143,0.005617,0.008090,0.020813,-0.000451,0.011095,0.008460,0.032839,0.101519,0.066496,0.002932,0.004101,0.012621,0.040790,0.000611,0.000598,0.001357,0.000852,0.000850,14.986980,18.063054,-3.076074,1.803075,-1.003064,0.170977,0.118168,0.089054,0.115110,0.135439,0.153778,0.159867,0.135519,0.118286,0.121145,1.919929,...,0.001348,3.538781e-02,2.230769e-05,2.230769e-05,3.538781e-02,1.348040e-03,9.632418e-01,3.541012e-02,0.000000e+00,9.645899e-01,0.000000e+00,0.0,3.541012e-02,0.964590,3.541012e-02,0.964590,0.964590,2.230769e-05,3.538781e-02,0.001348,9.632418e-01,-0.292809,-0.007434,-0.835767,-0.897188,0.131217,-0.389067,0.430878,0.009383,0.004029,0.002184,0.003187,0.767296,0.001305,0.005613,0,15,1.0,0.85,17.0,0,0.0,0.0,0.0,1.0,0.0,3.0,2.0,0,1.0,0.0,0.0,0.0,0,0.0,0.0,0.0,1.0,0,0.0,0.0,1.0,0.0,0.0,0.963242,-0.000057,0.005406,0.035410,0.035410,0.964590
5054,2022-02-02,nq1s,-1,11407.322107,11506.096593,11178.104980,11396.389053,8.559580e+05,3.130390e+05,0.007954,0.007986,0.014056,0.047218,0.067521,0.005388,-0.071348,-0.053021,0.003190,0.117573,1.0,-1.0,-1.0,1.0,1.0,0.067521,-0.071348,-0.053021,0.138870,-0.018328,0.029121,0.038785,0.003419,-0.039225,-0.031055,0.006560,0.019476,0.021556,0.005012,-0.021457,-0.024975,-0.075097,-0.042498,0.008503,0.047654,0.038821,-0.014187,-0.022137,-0.026337,-0.026176,-0.003840,-0.001853,-0.000205,0.000528,-0.0008

## 10. Build the feature catalog

The catalog records the feature source and group so later feature selection results are easier to interpret.

In [10]:
raw_cols = ["open", "high", "low", "close", "volume", "open_interest"]
hmm_latent_metadata_cols = ["hmm_predicted_regime", "hmm_regime_confidence"]
hmm_category_label_cols = []
hmm_category_code_cols = ["hmm_volatility_regime_code", "hmm_return_regime_code", "hmm_market_state_code"]
hmm_category_probability_cols = [col for col in hmm_category_merge.columns if col.startswith("hmm_prob_")]

catalog_rows = []

def add_catalog_rows(columns, group, source, description):
    for col in columns:
        catalog_rows.append(
            {
                "feature_name": col,
                "feature_group": group,
                "description": description,
                "source": source,
            }
        )

add_catalog_rows(raw_cols, "raw_market_data", "data/raw/ohlcv_data.csv", "Original OHLCV/open-interest field.")
add_catalog_rows(["primary_signal"], "primary_signal", "data/raw/primary_signals.csv", "Primary model signal reshaped to long format.")
add_catalog_rows(technical_feature_cols, "technical", "code/features/technical_analysis.py", "Generated technical-analysis feature from full OHLCV history.")
add_catalog_rows(hmm_latent_metadata_cols, "hmm_latent", "data/hmm/probabilities/all_category_probabilities.csv", "HMM latent regime id or confidence feature.")
add_catalog_rows(latent_probability_cols, "hmm_latent", "data/hmm/probabilities/latent_regime_probabilities.csv", "HMM latent regime probability feature.")
add_catalog_rows(hmm_category_code_cols, "hmm_category_code", "data/features/categorical_feature_mappings.csv", "Numeric code for an interpreted HMM category label.")
add_catalog_rows(hmm_category_probability_cols, "hmm_category_probability", "data/hmm/probabilities/all_category_probabilities.csv", "Interpreted HMM category probability feature.")
add_catalog_rows(market_feature_cols, "engineered_market", "notebooks/03_features/feature_engineering_matrix.ipynb", "New backward-looking OHLCV, volume, open-interest, candle, or volatility feature.")
add_catalog_rows(signal_feature_cols, "engineered_signal", "notebooks/03_features/feature_engineering_matrix.ipynb", "New backward-looking primary-signal behaviour feature.")
add_catalog_rows(hmm_encoding_feature_cols, "engineered_hmm_encoding", "notebooks/03_features/feature_engineering_matrix.ipynb", "Nullable one-hot or missing indicator derived from HMM labels.")
add_catalog_rows(interaction_feature_cols, "engineered_interaction", "notebooks/03_features/feature_engineering_matrix.ipynb", "New HMM regime or HMM interaction feature.")

feature_catalog = pd.DataFrame(catalog_rows).drop_duplicates("feature_name").sort_values(["feature_group", "feature_name"])

print(feature_catalog.shape)
display(feature_catalog.head(20))

(188, 4)


,feature_name,feature_group,description,source
180,hmm_market_calm_negative,engineered_hmm_encoding,Nullable one-hot or missing indicator derived ...,notebooks/03_features/feature_engineering_matr...
179,hmm_market_calm_positive,engineered_hmm_encoding,Nullable one-hot or missing indicator derived ...,notebooks/03_features/feature_engineering_matr...
181,hmm_market_chop,engineered_hmm_encoding,Nullable one-hot or missing indicator derived ...,notebooks/03_features/feature_engineering_matr...
176,hmm_market_state_missing,engineered_hmm_encoding,Nullable one-hot or missing indicator derived ...,notebooks/03_features/feature_engineering_matr...
177,hmm_market_stress,engineered_hmm_encoding,Nullable one-hot or missing indicator derived ...,notebooks/03_features/feature_engineering_matr...
178,hmm_market_upside_breakout,engineered_hmm_encoding,Nullable one-hot or missing indicator derived ...,notebooks/03_features/feature_engineering_matr...
162,hmm_regime_0,engineered_hmm_encoding,Nullable one-hot or missing indicator derived ...,notebooks/03_features/feature_engineering_matr...
163,hmm_regime_1,engineered_hmm_encoding,Nullable one-hot or missing indicator derived ...,notebooks/03_features/feature_engineering_matr...
164,hmm_regime_2,engineered_hmm_encoding,Nullable one-hot or missing indicator derived ...,notebooks/03_features/feature_engineering_matr...
165,hmm_regime_3,engineered_hmm_encoding,Nullable one-hot or missing indicator derived ...,notebooks/03_features/feature_engineering_matr...


## 11. Export

These CSVs are the handoff to the later CPCV/feature-selection stage. The matrix remains unstandardized; scaling should happen inside CPCV folds if a model requires it.

In [11]:
feature_matrix.to_csv(MERGED_FEATURE_MATRIX_PATH, index=False)
feature_catalog.to_csv(FEATURE_CATALOG_PATH, index=False)
categorical_mapping_log.to_csv(CATEGORICAL_MAPPING_LOG_PATH, index=False)

print(f"Saved all TA features: {ALL_TECHNICAL_FEATURES_PATH} ({technical_features.shape[0]} rows x {technical_features.shape[1]} columns)")
print(f"Saved feature matrix: {MERGED_FEATURE_MATRIX_PATH} ({feature_matrix.shape[0]} rows x {feature_matrix.shape[1]} columns)")
print(f"Saved feature catalog: {FEATURE_CATALOG_PATH} ({feature_catalog.shape[0]} rows)")
print(f"Saved categorical mapping log: {CATEGORICAL_MAPPING_LOG_PATH} ({categorical_mapping_log.shape[0]} rows)")

Saved all TA features: C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\TestBranchAssignment\systematic-strategies-with-machine-learning-Cw\data\features\all_technical_features.csv (83547 rows x 110 columns)
Saved feature matrix: C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\TestBranchAssignment\systematic-strategies-with-machine-learning-Cw\data\features\merged_feature_matrix.csv (7095 rows x 190 columns)
Saved feature catalog: C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\TestBranchAssignment\systematic-strategies-with-machine-learning-Cw\data\features\feature_catalog.csv (188 rows)
Saved categorical mapping log: C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\TestBranchAssignment\systematic-strategies-with-machine-learning-Cw\data\features\categorical_feature_mappings.csv (13 rows)
